In [23]:
import pandas as pd
import numpy as np
import re
import requests
from sklearn.neighbors import BallTree
from sklearn.preprocessing import LabelEncoder
from dotenv import load_dotenv
import os
import zipfile

# **1. Load data**

*   Historical HDB data: `ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv`
*   Nearest 3 amenities for every HDB listing in Singapore: `hdb_amenity_table_full.csv`
*   Join historical HDB data (`df_historical`) to amenity table (`df_amenities`) to get the 3 nearest amenities for every historical HDB transaction



In [24]:
# Load data

## Historical HDB data
df_historical = pd.read_csv("../../datasets/ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv") # historical HDB data
df_amenities = pd.read_csv("../../datasets/hdb_amenity_table_full.csv") # nearest 3 amenities to each HDB block 
df_trains = pd.read_csv("../../datasets/train_station_coords.csv")
df_bus = pd.read_csv("../../datasets/bus_stop_coords.csv")
df_schools = pd.read_csv("../../datasets/Generalinformationofschools_with_coords.csv")
df_polyclinics = pd.read_csv("../../datasets/singapore_polyclinics_with_coords.csv")
df_hawkers = pd.read_csv("../../datasets/hawker_centres_final.csv")
df_supermarkets = pd.read_csv("../../datasets/supermarkets_coords_clean.csv")
df_malls = pd.read_csv("../../datasets/sg_malls_final.csv")
df_rpi = pd.read_csv("../../datasets/HDBResalePriceIndex1Q2009100Quarterly.csv")

In [25]:
# Join amenities data to historical dataset
merged_df = pd.merge(
    df_historical,
    df_amenities,
    left_on = ["block", "street_name"],
    right_on = ["blk_no", "street"],
    how = "left"
)

merged_df.drop(["blk_no", "street", "town_y"], axis = 1, inplace = True)
merged_df.rename(columns = {"town_x": "town"}, inplace = True)
merged_df

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,...,train_2_name,train_2_dist_m,train_3_name,train_3_dist_m,bus_1_name,bus_1_dist_m,bus_2_name,bus_2_dist_m,bus_3_name,bus_3_dist_m
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,...,Bishan,1300.0,Lorong Chuan,1625.0,54319.0,98.0,54311.0,154.0,54371.0,210.0
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,...,Bright Hill,1038.0,Ang Mo Kio,1268.0,54189.0,169.0,54181.0,191.0,54221.0,196.0
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,...,Mayflower,1037.0,Yio Chu Kang,1071.0,55129.0,138.0,55121.0,144.0,55011.0,154.0
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,...,Lorong Chuan,1797.0,Bishan,1882.0,54389.0,72.0,54381.0,129.0,54299.0,323.0
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,...,Mayflower,1077.0,Yio Chu Kang,1094.0,55011.0,124.0,55019.0,154.0,55129.0,179.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
228121,2026-03,YISHUN,EXECUTIVE,643,YISHUN ST 61,10 TO 12,142.0,Apartment,1987,60 years 06 months,...,Yishun,941.0,Canberra,2566.0,59369.0,109.0,59371.0,147.0,59361.0,158.0
228122,2026-03,YISHUN,EXECUTIVE,611,YISHUN ST 61,07 TO 09,146.0,Maisonette,1987,60 years 09 months,...,Yishun,1036.0,Canberra,2643.0,59059.0,135.0,59371.0,190.0,59051.0,212.0
228123,2026-03,YISHUN,EXECUTIVE,877,YISHUN ST 81,10 TO 12,142.0,Apartment,1987,60 years 10 months,...,Yishun,1729.0,Springleaf,2668.0,59499.0,122.0,59481.0,205.0,59721.0,281.0
228124,2026-03,YISHUN,EXECUTIVE,836,YISHUN ST 81,10 TO 12,146.0,Maisonette,1988,61 years,...,Yishun,1570.0,Springleaf,2611.0,59041.0,108.0,59049.0,160.0,59039.0,291.0


# **2. Data cleaning & processing**

*   Handle NAs
    * Identified rows with missing amenities distances 
    * Imputed missing values using OneMap API (to get coordinates and address) and haversine distance (to get amenity distances)

*   Standardise column data types






**2a. Handle NAs**

In [26]:
# Config & Helper functions

## 1. Config: Amenity data sources and their column names (for easy lookup during imputation)

### Each entry follows the format (prefix, df_name, lat_col, lon_col, name_col)
AMENITY_CONFIG = [
    ("mall", "df_malls", "lat", "lon", "mall"),
    ("school", "df_schools", "lat", "long", "school_name"),
    ("hawker", "df_hawkers", "lat", "lng", "NAME"),
    ("polyclinic", "df_polyclinics", "lat", "long", "name"),
    ("supermarket", "df_supermarkets", "lat", "lng", "name"),
    ("train", "df_trains", "lat", "lng", "stn_name"),
    ("bus", "df_bus", "Latitude", "Longitude", "BUS_STOP_NUM")
]

amenity_dfs = {
    "df_malls": df_malls,
    "df_schools": df_schools,
    "df_hawkers": df_hawkers,
    "df_polyclinics": df_polyclinics,
    "df_supermarkets": df_supermarkets,
    "df_trains": df_trains,
    "df_bus": df_bus,
}

## 2. Geocoding helper functions

### Authenticate OneMap API Token
load_dotenv(dotenv_path = "../.env")
auth_url = "https://www.onemap.gov.sg/api/auth/post/getToken"
ONEMAP_EMAIL = os.getenv("ONEMAP_EMAIL")
ONEMAP_PW = os.getenv("ONEMAP_PW")

payload = {
    "email": ONEMAP_EMAIL,
    "password": ONEMAP_PW
}

response = requests.request("POST", auth_url, json=payload)
token = response.json()["access_token"]

### Get coordinates of an address
def search_coords(search_val):
  search_url = f"https://www.onemap.gov.sg/api/common/elastic/search?searchVal={search_val}&returnGeom=Y&getAddrDetails=Y&pageNum=1"
  headers = {"Authorization": token}
  response = requests.get(search_url, headers=headers)
  return response.json()

## 3. Distance helper functions 

### Get lat, lon, full_address, postal for given address string using OneMap API
def get_address_info(address):
  search_result = search_coords(address)
  if not search_result or search_result.get("found") == 0:
    print(f"No results found for address: {address}")
    return None
  else:
    return {
        "lat": float(search_result["results"][0]["LATITUDE"]),
        "lon": float(search_result["results"][0]["LONGITUDE"]),
        "full_address": search_result["results"][0]["ADDRESS"],
        "postal": search_result["results"][0]["POSTAL"]
    }

### Vectorised function to get haversine distance (in m) from one point to an array of points
def haversine_distance_vectorised(lat0, lon0, lats, lons):
  R = 6371000 # earth's radius in m
  phi0, phi1 = np.radians(lat0), np.radians(lats)
  dphi = phi1 - phi0
  dlambda = np.radians(lons - lon0)
  a = np.sin(dphi/2)**2 + np.cos(phi0) * np.cos(phi1) * np.sin(dlambda/2)**2
  c = 2 * np.arcsin(np.sqrt(a))
  return R * c

### Get k nearest amenities to one point as a list of dictionaries following the format {name, dist_m}
def get_nearest_k_amenities(lat, lon, amenity_df, lat_col, lon_col, name_col, k):
  distances = haversine_distance_vectorised(lat, lon, amenity_df[lat_col].to_numpy(), amenity_df[lon_col].to_numpy())
  topk_partial_sort = np.argpartition(distances, k)[:k] # partial sort for efficiency
  topk_sorted = topk_partial_sort[np.argsort(distances[topk_partial_sort])] # top k fully sorted
  return [
      {"name": amenity_df.iloc[i][name_col],
       "dist_m": round(distances[i])}
      for i in topk_sorted
  ]

### Build the complete impute dictionary for a single address
def build_impute_dict(lat, lon, full_address, postal, amenity_dfs):
  impute_dict = {
      "lat": lat,
      "lon": lon,
      "full_address": full_address,
      "postal": postal
  }

  for prefix, df_name, lat_col, lon_col, name_col in AMENITY_CONFIG:
    nearest = get_nearest_k_amenities(lat, lon, amenity_dfs[df_name], lat_col, lon_col, name_col, 3)
    for rank, item in enumerate(nearest, 1):
      impute_dict[f"{prefix}_{rank}_name"] = item["name"]
      impute_dict[f"{prefix}_{rank}_dist_m"] = item["dist_m"]

  return impute_dict

In [27]:
# 1. Identify rows with postal == "NIL" and impute the missing postal codes
missing_postal_rows = merged_df[merged_df["postal"] == "NIL"]

if missing_postal_rows.empty:
  print("No rows with postal == NIL")
else:
  print(f"{len(missing_postal_rows)} rows with postal == NIL:")
  print(missing_postal_rows[["block", "street_name", "postal"]])

  merged_df.loc[merged_df["block"] == "215", "postal"] = "680215" # checked postal codes manually
  merged_df.loc[merged_df["block"] == "216", "postal"] = "680216"

print(f"Remaining rows with postal == NIL: {len(merged_df[merged_df["postal"] == "NIL"])}")
print("-" * 80)

# 2. Identify rows with missing (NA) values and impute the missing data
na_rows = merged_df[merged_df.isna().any(axis=1)]

if na_rows.empty:
  print("No rows with NaN values found")
else:
  print("Rows with NaN values:")
  print(na_rows[["block", "street_name"]])

  for (block, street), _ in na_rows.groupby(["block", "street_name"]):
    address = f"{block} {street}"
    coords = get_address_info(address) # get block-level address info (lat, lon, full_address, postal)

    if coords is None:
      print(f"Could not find address in OneMap. Skipping this address.")
      continue

    ## Get amenity distance info
    impute_values = build_impute_dict(coords["lat"], coords["lon"], coords["full_address"], coords["postal"], amenity_dfs)
    impute_mask = (
        (merged_df["block"] == block) &
        (merged_df["street_name"] == street)
    )
    for col, val in impute_values.items():
      merged_df.loc[impute_mask, col] = val

    print(f"Successfully imputed {len(impute_values)} columns for {address}")

print("-" * 80)

print(f"Rows remaining: {len(merged_df)}")
print(f"Missing values remaining: {merged_df.isna().sum().sum()}")

15 rows with postal == NIL:
       block         street_name postal
1458     216  CHOA CHU KANG CTRL    NIL
21872    215  CHOA CHU KANG CTRL    NIL
66588    215  CHOA CHU KANG CTRL    NIL
71193    216  CHOA CHU KANG CTRL    NIL
92845    215  CHOA CHU KANG CTRL    NIL
149668   216  CHOA CHU KANG CTRL    NIL
150062   215  CHOA CHU KANG CTRL    NIL
150432   215  CHOA CHU KANG CTRL    NIL
150554   215  CHOA CHU KANG CTRL    NIL
176141   215  CHOA CHU KANG CTRL    NIL
176147   216  CHOA CHU KANG CTRL    NIL
176151   216  CHOA CHU KANG CTRL    NIL
176623   215  CHOA CHU KANG CTRL    NIL
176725   216  CHOA CHU KANG CTRL    NIL
203160   215  CHOA CHU KANG CTRL    NIL
Remaining rows with postal == NIL: 0
--------------------------------------------------------------------------------
Rows with NaN values:
      block      street_name
390      82  MACPHERSON LANE
13989    82  MACPHERSON LANE
15726    82  MACPHERSON LANE
Successfully imputed 46 columns for 82 MACPHERSON LANE
---------------------

**2b. Standardise column data types**

In [28]:
merged_df.info()

# Standardise data types
merged_df["month"] = pd.to_datetime(merged_df["month"], format="%Y-%m")
merged_df["postal"] = pd.to_numeric(merged_df["postal"], errors="coerce").astype("Int64") ## Postal as nullable integer (values may have trailing .0 from CSV round-trip)

categorical_cols = ["town", "flat_type", "flat_model"]
for col in categorical_cols:
    merged_df[col] = merged_df[col].astype("category")

# Check category values for categorical variables
for col in categorical_cols:
    print(f"Column: {col}")
    print(f"Total Unique Categories: {len(merged_df[col].cat.categories)}")
    print(merged_df[col].cat.categories.tolist())
    print("-" * 80)

<class 'pandas.DataFrame'>
RangeIndex: 228126 entries, 0 to 228125
Data columns (total 57 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   month                 228126 non-null  str    
 1   town                  228126 non-null  str    
 2   flat_type             228126 non-null  str    
 3   block                 228126 non-null  str    
 4   street_name           228126 non-null  str    
 5   storey_range          228126 non-null  str    
 6   floor_area_sqm        228126 non-null  float64
 7   flat_model            228126 non-null  str    
 8   lease_commence_date   228126 non-null  int64  
 9   remaining_lease       228126 non-null  str    
 10  resale_price          228126 non-null  float64
 11  lat                   228126 non-null  float64
 12  lon                   228126 non-null  float64
 13  postal                228126 non-null  str    
 14  full_address          228126 non-null  str    
 15  mall_1_name

# **3. Feature engineering**

* Property features:
  * `storey_midpoint`: Extracted `storey_start` and `storey_end` from `storey_range`, then calculated the midpoint

* Distance features
  * `num_mrt_within_1km`
  * `flag_mrt_within_500m`
  * `num_bus_within_400m`
  * `num_hawkers_within_500m`
  * `num_primary_schs_within_1km`
  * `dist_cbd`
  
* Temporal/market features:
  * `rpi`: Resale Price Index - tracks overall price movement of the HDB market, calculated using resale transactions registered across towns, flat types and models, with the first quarter of 2009 as the base period
  * `month_index`: number of months since Jan 2017


In [29]:
# Property features

## storey_midpoint: Parse storey_range
merged_df["storey_start"] = merged_df["storey_range"].str.split(" TO ").str[0].astype(int)
merged_df["storey_end"] = merged_df["storey_range"].str.split(" TO ").str[1].astype(int)
merged_df["storey_midpoint"] = (merged_df["storey_start"] + merged_df["storey_end"]) / 2

## remaining_lease: Parse remaining_lease from string to float (years)
def convert_lease_to_float(lease_str):
    if pd.isna(lease_str):
        return None

    years = 0
    months = 0

    year_match = re.search(r"(\d+)\s+years?", str(lease_str)) # look for digits before "year(s)"
    if year_match:
        years = int(year_match.group(1))

    month_match = re.search(r"(\d+)\s+months?", str(lease_str)) # look for digits before "month(s)"
    if month_match:
        months = int(month_match.group(1))

    return round(years + (months / 12), 3) # convert to years

merged_df["remaining_lease"] = merged_df["remaining_lease"].apply(convert_lease_to_float)

In [30]:
# Helper functions for distance features
def count_within_radius(block_lats, block_lons, amenity_lats, amenity_lons, radius_m):
  R = 6_371_000  # earth's radius (m)

  # Reshape 
  lat0 = np.radians(block_lats[:, None])
  lon0 = np.radians(block_lons[:, None])
  lat1 = np.radians(amenity_lats[None, :])
  lon1 = np.radians(amenity_lons[None, :])

  dlat = lat1 - lat0
  dlon = lon1 - lon0
  a = np.sin(dlat / 2) ** 2 + np.cos(lat0) * np.cos(lat1) * np.sin(dlon / 2) ** 2
  dist = R * 2 * np.arcsin(np.sqrt(a)) # shape: (n_blocks, n_amenities)

  return (dist <= radius_m).sum(axis=1).astype(int)

## Faster for large datasets 
def count_within_radius_balltree(block_lats, block_lons, amenity_lats, amenity_lons, radius_m):
    R = 6_371_000
    amenity_coords = np.radians(
        np.column_stack([amenity_lats, amenity_lons])
    )
    block_coords = np.radians(
        np.column_stack([block_lats, block_lons])
    )
    tree = BallTree(amenity_coords, metric="haversine")
    counts = tree.query_radius(
        block_coords,
        r=radius_m / R, # radians
        count_only=True
    )
    return counts.astype(int)

In [31]:
# Distance features
df_primary_schs = df_schools[df_schools["mainlevel_code"] == "PRIMARY"].dropna(subset=["lat", "long"])
df_trains = df_trains.dropna(subset=["lat", "lng"])
df_bus = df_bus.dropna(subset=["Latitude", "Longitude"])
df_hawkers = df_hawkers.dropna(subset=["lat", "lng"])

block_lats = merged_df["lat"].to_numpy()
block_lons = merged_df["lon"].to_numpy()

## num_mrt_within_1km
MRT_COUNT_THRESHOLD = 1000
merged_df["num_mrt_within_1km"] = count_within_radius(block_lats, block_lons, df_trains["lat"].to_numpy(), df_trains["lng"].to_numpy(),
    MRT_COUNT_THRESHOLD)

## flag_mrt_within_500m
MRT_FLAG_THRESHOLD = 500
merged_df["flag_mrt_within_500m"] = (
    count_within_radius(
        block_lats, block_lons,
        df_trains["lat"].to_numpy(), df_trains["lng"].to_numpy(),
        MRT_FLAG_THRESHOLD
    ) > 0
).astype(int)

## num_primary_schools_within_1km
SCHOOL_COUNT_THRESHOLD = 1000
merged_df["num_primary_schools_within_1km"] = count_within_radius(
    block_lats, block_lons,
    df_primary_schs["lat"].to_numpy(), df_primary_schs["long"].to_numpy(),
    SCHOOL_COUNT_THRESHOLD,
)

## num_hawkers_within_500m
HAWKER_COUNT_THRESHOLD = 500
merged_df["num_hawkers_within_500m"] = count_within_radius(
    block_lats, block_lons,
    df_hawkers["lat"].to_numpy(), df_hawkers["lng"].to_numpy(),
    HAWKER_COUNT_THRESHOLD,
)

## num_bus_within_400m
BUS_COUNT_THRESHOLD = 400
merged_df["num_bus_within_400m"] = count_within_radius_balltree(
    block_lats, block_lons,
    df_bus["Latitude"].to_numpy(), df_bus["Longitude"].to_numpy(),
    BUS_COUNT_THRESHOLD,
)

COUNT_FEATURES = [
    "num_mrt_within_1km",
    "flag_mrt_within_500m",
    "num_bus_within_400m",
    "num_primary_schools_within_1km",
    "num_hawkers_within_500m"
]

## Check
print("Feature summary:")
print(merged_df[COUNT_FEATURES].describe().round(2))
print("Missing values:", merged_df[COUNT_FEATURES].isna().sum().to_dict())

Feature summary:
       num_mrt_within_1km  flag_mrt_within_500m  num_bus_within_400m  \
count           228126.00             228126.00            228126.00   
mean                 2.38                  0.45                10.15   
std                  2.28                  0.50                 3.39   
min                  0.00                  0.00                 1.00   
25%                  1.00                  0.00                 8.00   
50%                  2.00                  0.00                10.00   
75%                  3.00                  1.00                12.00   
max                 14.00                  1.00                23.00   

       num_primary_schools_within_1km  num_hawkers_within_500m  
count                       228126.00                228126.00  
mean                             2.99                     0.54  
std                              1.57                     0.77  
min                              0.00                     0.00  
25%      

In [32]:
# dist_cbd

## Get coordinates for Raffles Place MRT station (proxy for CBD)
cbd_coords = df_trains.loc[df_trains["stn_name"] == "Raffles Place", ["lat", "lng"]].iloc[0]
cbd_lat, cbd_lon = cbd_coords["lat"], cbd_coords["lng"]

## Calculate haversine distances from each HDB to CBD
merged_df["dist_cbd"] = haversine_distance_vectorised(cbd_lat, cbd_lon, merged_df["lat"].to_numpy(), merged_df["lon"].to_numpy())
merged_df

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,...,bus_3_dist_m,storey_start,storey_end,storey_midpoint,num_mrt_within_1km,flag_mrt_within_500m,num_primary_schools_within_1km,num_hawkers_within_500m,num_bus_within_400m,dist_cbd
0,2017-01-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61.333,...,210.0,10,12,11.0,0,0,2,1,10,8663.913198
1,2017-01-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60.583,...,196.0,1,3,2.0,1,1,2,3,10,9768.106654
2,2017-01-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62.417,...,154.0,1,3,2.0,1,0,1,0,6,10887.555476
3,2017-01-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62.083,...,323.0,4,6,5.0,1,0,3,2,7,9148.645081
4,2017-01-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62.417,...,179.0,1,3,2.0,1,0,1,0,6,10928.364849
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
228121,2026-03-01,YISHUN,EXECUTIVE,643,YISHUN ST 61,10 TO 12,142.0,Apartment,1987,60.500,...,158.0,10,12,11.0,2,0,4,0,12,15336.454532
228122,2026-03-01,YISHUN,EXECUTIVE,611,YISHUN ST 61,07 TO 09,146.0,Maisonette,1987,60.750,...,212.0,7,9,8.0,1,1,3,0,13,15226.277355
228123,2026-03-01,YISHUN,EXECUTIVE,877,YISHUN ST 81,10 TO 12,142.0,Apartment,1987,60.833,...,281.0,10,12,11.0,1,1,2,0,7,14539.801915
228124,2026-03-01,YISHUN,EXECUTIVE,836,YISHUN ST 81,10 TO 12,146.0,Maisonette,1988,61.000,...,291.0,10,12,11.0,1,1,2,0,10,14744.951102


In [33]:
# Temporal features
## RPI
df_rpi.rename(columns={"index": "rpi"}, inplace=True)
df_rpi["rpi"] = pd.to_numeric(df_rpi["rpi"])
df_rpi.loc[len(df_rpi)] = { # Flash estimate for 2026-Q1 from HDB (actual data has not been released)
    "quarter": "2026-Q1", 
    "rpi": 203.4
 }  

### Drop if already exists (prevents re-run errors)
merged_df.drop(columns=["rpi", "quarter_key"], inplace=True, errors="ignore")

merged_df["quarter_key"] = (
    merged_df["month"].dt.year.astype(str) + "-Q" +
    merged_df["month"].dt.quarter.astype(str)
)

### Extrapolate missing quarters using RPI of last 4 quarters
missing_quarters = merged_df["quarter_key"][merged_df["quarter_key"].isin(df_rpi["quarter"]) == False].unique()

if len(missing_quarters) > 0:
    print(f"Quarters in transaction data with no RPI: {sorted(missing_quarters)}")

    recent = df_rpi.tail(4).copy().reset_index(drop=True)
    recent["t"] = range(4)
    slope, intercept = np.polyfit(recent["t"], recent["rpi"], deg=1)

    last_quarter = df_rpi["quarter"].iloc[-1] 
    last_t = len(df_rpi) - 1

    extrapolated_rows = []
    for q in sorted(missing_quarters):
        ### Compute how many quarters ahead of the last known quarter this is
        q_year, q_num = int(q.split("-Q")[0]), int(q.split("-Q")[1])
        lq_year, lq_num = int(last_quarter.split("-Q")[0]), int(last_quarter.split("-Q")[1])
        steps_ahead = (q_year - lq_year) * 4 + (q_num - lq_num)
        t_extrap = 3 + steps_ahead
        rpi_extrap = intercept + slope * t_extrap
        extrapolated_rows.append({"quarter": q, "rpi": round(rpi_extrap, 1)})
        print(f"Extrapolated RPI for {q}: {rpi_extrap:.1f} (last known: {df_rpi["rpi"].iloc[-1]}, slope: {slope:+.2f}/quarter)")

    df_rpi = pd.concat([df_rpi, pd.DataFrame(extrapolated_rows)], ignore_index=True)
else:
    print("All quarters covered by RPI data — no extrapolation needed.")

### Join RPI value for that quarter
merged_df = pd.merge(merged_df, df_rpi, left_on="quarter_key", right_on="quarter", how="left")
merged_df.drop(columns=["quarter"], inplace=True)

print("Missing RPI values:", merged_df["rpi"].isna().sum())
merged_df[["month", "quarter_key", "rpi"]].drop_duplicates().sort_values("month").head(10)

### Calculate real price 
RPI_BASE = 100.0 # Q1 2009 index value (the base period)
 
merged_df["real_price"] = merged_df["resale_price"] / merged_df["rpi"] * RPI_BASE
 
### Store the current RPI for latest known quarter (for price conversion)
RPI_CURRENT = df_rpi["rpi"].iloc[-1]
print(f"Base RPI: {RPI_BASE}")
print(f"Current RPI: {RPI_CURRENT:.1f} (used to convert predictions back to nominal value)")

# Number of months since Jan 2017
merged_df["month_index"] = (
    (merged_df["month"].dt.year - 2017) * 12 + (merged_df["month"].dt.month - 1)
)

All quarters covered by RPI data — no extrapolation needed.
Missing RPI values: 0
Base RPI: 100.0
Current RPI: 203.4 (used to convert predictions back to nominal value)


In [40]:
# Prepare Features for Model
## Define columns to drop
drop_cols = [
    "month", # replaced by month_index
    "storey_range", # replaced by storey_midpoint
    "storey_start", # redundant
    "storey_end", # redundant
    "quarter_key", # used for RPI merge, not needed as feature
    "flat_model",
    "resale_price" # replaced by real_price
]

## Drop amenity name columns (only need distances)
amenity_name_cols = [c for c in merged_df.columns if ("_name" in c and c != "street_name")]
print("Dropping amenity name cols:", amenity_name_cols)

feature_df = merged_df.drop(columns=drop_cols + amenity_name_cols, errors="ignore")

## Drop remaining NAs first (before copying)
print(f"Rows before dropping NAs: {len(feature_df)}")
feature_df.dropna(inplace=True)
print(f"Rows after dropping NAs: {len(feature_df)}")

## Two copies: raw for XGBoost/CatBoost, encoded for Random Forest
cat_cols = ["town", "flat_type"]

### Raw copy: categorical columns keep their string values
feature_df_raw = feature_df.copy().drop(columns=["postal", "block", "street_name", "full_address"]) # too granular for model

### Encoded copy: categorical columns label-encoded to integers
feature_df_enc = feature_df.copy().drop(columns=["postal", "block", "street_name", "full_address"])
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    feature_df_enc[col] = le.fit_transform(feature_df_enc[col].astype(str))
    label_encoders[col] = le

print("Label encoding complete.")
print(f"dtype of town from feature_df_raw: {feature_df_raw["town"].dtype}") # should be object
print(f"dtype of town from feature_df_enc: {feature_df_enc["town"].dtype}") # should be int64
print(f"\nFinal features ({len(feature_df_raw.columns)}): {feature_df_raw.columns.tolist()}")

Dropping amenity name cols: ['mall_1_name', 'mall_2_name', 'mall_3_name', 'school_1_name', 'school_2_name', 'school_3_name', 'hawker_1_name', 'hawker_2_name', 'hawker_3_name', 'polyclinic_1_name', 'polyclinic_2_name', 'polyclinic_3_name', 'supermarket_1_name', 'supermarket_2_name', 'supermarket_3_name', 'train_1_name', 'train_2_name', 'train_3_name', 'bus_1_name', 'bus_2_name', 'bus_3_name']
Rows before dropping NAs: 228126
Rows after dropping NAs: 228126
Label encoding complete.
dtype of town from feature_df_raw: category
dtype of town from feature_df_enc: int64

Final features (38): ['town', 'flat_type', 'floor_area_sqm', 'lease_commence_date', 'remaining_lease', 'lat', 'lon', 'mall_1_dist_m', 'mall_2_dist_m', 'mall_3_dist_m', 'school_1_dist_m', 'school_2_dist_m', 'school_3_dist_m', 'hawker_1_dist_m', 'hawker_2_dist_m', 'hawker_3_dist_m', 'polyclinic_1_dist_m', 'polyclinic_2_dist_m', 'polyclinic_3_dist_m', 'supermarket_1_dist_m', 'supermarket_2_dist_m', 'supermarket_3_dist_m', 'train

In [42]:
# Save final feature_dfs to csv
feature_df.to_csv("../csv_outputs/feature_df.csv", index=False)
feature_df_raw.to_csv("../csv_outputs/feature_df_raw.csv", index=False)
feature_df_enc.to_csv("../csv_outputs/feature_df_enc.csv", index=False)
with zipfile.ZipFile("../csv_outputs/feature_df.zip", "w", zipfile.ZIP_DEFLATED, compresslevel=9) as zf:
    zf.write("../csv_outputs/feature_df.csv", "feature_df.csv")
with zipfile.ZipFile("../csv_outputs/feature_df_raw.zip", "w", zipfile.ZIP_DEFLATED, compresslevel=9) as zf:
    zf.write("../csv_outputs/feature_df_raw.csv", "feature_df_raw.csv")
with zipfile.ZipFile("../csv_outputs/feature_df_enc.zip", "w", zipfile.ZIP_DEFLATED, compresslevel=9) as zf:
    zf.write("../csv_outputs/feature_df_enc.csv", "feature_df_enc.csv")